# Train the "Hey FRIDAY" wake word (Google Colab)

Trains a custom [openWakeWord](https://github.com/dscripka/openWakeWord) model for the phrase **"hey friday"** and exports `hey_friday.onnx`. Drop that file into FRIDAY at `models/wakeword/hey_friday.onnx` and point the wake-word seam at it (`FRIDAY_WAKEWORD_MODEL=models/wakeword/hey_friday.onnx`).

**Runtime:** Colab → *Runtime ▸ Change runtime type ▸ GPU* (T4 is plenty). The whole run is ~15-30 min.

**How it works (openWakeWord custom-model recipe):**
1. Synthesize many spoken samples of "hey friday" with a TTS (piper-sample-generator) across varied speakers.
2. Mix in noise/reverb augmentation so the model is robust.
3. Use the shared openWakeWord audio-embedding model to turn clips into features.
4. Train a small classifier (positives = "hey friday", negatives = a large speech/noise corpus).
5. Export to ONNX and test it.

> If an API/URL below drifts, follow the upstream `automatic_model_training.ipynb` in the openWakeWord repo — this notebook mirrors it, specialized to "hey friday".

## 1. Install dependencies

In [ ]:
# openWakeWord (training extras) + the synthetic-speech generator + audio aug.
!pip install -q openwakeword==0.6.0 onnx onnxruntime
!pip install -q piper-phonemize piper-tts 2>/dev/null || true
!git clone -q https://github.com/rhasspy/piper-sample-generator.git
!pip install -q -r piper-sample-generator/requirements.txt
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
    https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt || true
import os; os.makedirs('out', exist_ok=True); print('ok')

## 2. Download the openWakeWord base models (shared feature extractor)

In [ ]:
import openwakeword
from openwakeword.utils import download_models
download_models()  # melspectrogram + embedding models the trainer/inference share
print('base models ready')

## 3. Synthesize positive samples of "hey friday"

`piper-sample-generator` produces many varied utterances of the phrase. Bump `--max-samples` for a stronger model (1000-2000 is a good range).

In [ ]:
WAKE_PHRASE = 'hey friday'
N_POSITIVES = 1500
!cd piper-sample-generator && python generate_samples.py \
    "{WAKE_PHRASE}" \
    --model models/en_US-libritts_r-medium.pt \
    --max-samples {N_POSITIVES} \
    --batch-size 64 \
    --output-dir ../out/positives
import glob; print('positive clips:', len(glob.glob('out/positives/*.wav')))

## 4. Get negatives + augmentation audio (background speech, noise, room impulses)

openWakeWord ships precomputed negative *features* and augmentation clips so you don't need to hand-collect hours of audio. This pulls the standard training bundle.

In [ ]:
# Precomputed negative features + RIRs + noise used by the upstream trainer.
# (See openWakeWord docs 'Training new models' for the current asset URLs.)
!mkdir -p out/negatives out/rirs out/noise
!wget -q -O out/negative_features.npy \
    https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy || \
    echo 'NOTE: set the negative-features URL from the openWakeWord docs if this 404s'
print('negatives staged')

## 5. Compute features for the positive clips

In [ ]:
import numpy as np, glob, scipy.io.wavfile as wav
from openwakeword.utils import AudioFeatures
fe = AudioFeatures()

def clip_to_features(path):
    sr, data = wav.read(path)
    if data.dtype != np.int16:
        data = (data * 32767).astype(np.int16)
    return fe.embed_clips([data], batch_size=1)[0]

pos = np.array([clip_to_features(p) for p in glob.glob('out/positives/*.wav')])
neg = np.load('out/negative_features.npy')
print('positive features', pos.shape, '| negative features', neg.shape)

## 6. Train the classifier

In [ ]:
import numpy as np, tensorflow as tf
from tensorflow.keras import layers, models

# Window the embedding sequences to the openWakeWord input shape (16 x 96).
WIN = 16
def windows(feats):
    out = []
    for seq in feats:
        for i in range(0, max(1, len(seq) - WIN)):
            out.append(seq[i:i+WIN])
    return np.array([w for w in out if w.shape == (WIN, 96)])

Xp, Xn = windows(pos), windows(neg if neg.ndim == 3 else neg[None])
X = np.concatenate([Xp, Xn]); y = np.concatenate([np.ones(len(Xp)), np.zeros(len(Xn))])

model = models.Sequential([
    layers.Input((WIN, 96)), layers.Flatten(),
    layers.Dense(128, activation='relu'), layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])
model.compile('adam', 'binary_crossentropy', metrics=['accuracy'])
model.fit(X, y, epochs=30, batch_size=256, validation_split=0.1,
          class_weight={0: 1.0, 1: len(Xn)/max(1, len(Xp))})

## 7. Export to ONNX

In [ ]:
!pip install -q tf2onnx
import tf2onnx, tensorflow as tf
spec = (tf.TensorSpec((None, WIN, 96), tf.float32, name='input'),)
tf2onnx.convert.from_keras(model, input_signature=spec, opset=13,
                           output_path='hey_friday.onnx')
print('exported hey_friday.onnx')

## 8. Quick sanity check + download

In [ ]:
import onnxruntime as ort, numpy as np
sess = ort.InferenceSession('hey_friday.onnx')
score = sess.run(None, {'input': Xp[:1].astype('float32')})[0]
print('score on a positive window (want ~1.0):', float(score[0][0]))
from google.colab import files  # noqa: E402
files.download('hey_friday.onnx')

## 9. Use it in FRIDAY

```bash
mkdir -p models/wakeword && mv ~/Downloads/hey_friday.onnx models/wakeword/
# .env
FRIDAY_ENABLE_WAKEWORD=true
FRIDAY_WAKEWORD_MODEL=models/wakeword/hey_friday.onnx
```

The server wake-word seam (`friday.voice.wake_engine`) lazy-loads this ONNX via openWakeWord, and on a "hey friday" detection pushes a wake event to the HUD over the existing voice WebSocket — which reveals the cockpit and has FRIDAY greet you in her voice.